# Neural networks: forward propagation

<a target="_blank" href="https://colab.research.google.com/github/changyaochen/MECE4520/blob/master/site/neural-networks/01-forward-propagation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

A neural network makes a prediction by repeatedly transforming its input. In
this notebook, we trace those transformations for a small gas-turbine
classification example. We use randomly generated weights: the goal is to
understand **forward propagation**, not yet to learn the weights from data.


## What you will do

1. Turn a small batch of gas-turbine observations into a feature matrix.
2. Compute a forward pass through one hidden layer with NumPy.
3. Inspect the intermediate scores, activations, and predicted probabilities.
4. See why an activation function changes what a stacked network can represent.
5. Count the parameters created by changing the hidden-layer width.


## 1. Set up a small prediction batch

Each row in the course dataset is one gas-turbine operating observation.
We use all nine measured operating features: AT, AP, AH, AFDP, GTEP, TIT,
TAT, TEY, and CDP. They illustrate a binary prediction of the positive
`high_co` label. `CO` itself is excluded because it defines that label.

We standardize the nine features so their numerical scales are comparable.
This notebook does not fit or evaluate a model, so the standardization only
makes the forward-pass calculation easy to inspect.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve

from collections.abc import Callable
from IPython.display import display

import numpy as np
import pandas as pd


In [2]:
DATA_URL = (
    "https://raw.githubusercontent.com/changyaochen/MECE4520/"
    "master/site/data/gas-turbine-course.csv"
)
FEATURES = ["AT", "AP", "AH", "AFDP", "GTEP", "TIT", "TAT", "TEY", "CDP"]
TARGET = "high_co"

candidates = [
    Path("../data/gas-turbine-course.csv"),
    Path("site/data/gas-turbine-course.csv"),
    Path("gas-turbine-course.csv"),
]
data_path = next((path for path in candidates if path.exists()), candidates[-1])

if not data_path.exists():
    urlretrieve(DATA_URL, data_path)

data = pd.read_csv(data_path)
model_data = data[FEATURES + [TARGET]].dropna().copy()

feature_mean = model_data[FEATURES].mean()
feature_std = model_data[FEATURES].std(ddof=0)
standardized_features = (model_data[FEATURES] - feature_mean) / feature_std

# Four fixed rows make the calculations below reproducible.
BATCH_ROWS = [13311, 4066, 8682, 5064]
X_batch = standardized_features.loc[BATCH_ROWS].to_numpy(dtype=float)
y_batch = model_data.loc[BATCH_ROWS, TARGET].to_numpy(dtype=int)

feature_summary = pd.DataFrame(
    {"mean": feature_mean, "standard deviation": feature_std}
)
feature_summary.index.name = "feature"

print(f"Feature matrix shape: {X_batch.shape}")
print(f"Label-vector shape: {y_batch.shape}")
print("\nAll input features and their training-data scales:")
feature_summary.round(3)


Feature matrix shape: (4, 9)
Label-vector shape: (4,)

All input features and their training-data scales:


,mean,standard deviation
feature,,
AT,17.715,7.441
AP,1013.071,6.477
AH,77.873,14.468
AFDP,3.924,0.775
GTEP,25.551,4.189
TIT,1081.396,17.557
TAT,546.180,6.821
TEY,133.466,15.591
CDP,12.058,1.087


The batch has $m=4$ observations and $p=9$ input features, so
$\mathbf{X}\in\mathbb{R}^{4\times9}$. We follow the usual NumPy
convention: **each observation occupies one row**.


## 2. Define a network with one hidden layer

Our network has three hidden units and one output unit. For a batch of inputs,
the forward pass is

$$
\begin{aligned}
\mathbf{Z}^{[1]} &= \mathbf{X}(\mathbf{W}^{[1]})^T+\mathbf{B}^{[1]},\\
\mathbf{A}^{[1]} &= \operatorname{ReLU}(\mathbf{Z}^{[1]}),\\
\mathbf{Z}^{[2]} &= \mathbf{A}^{[1]}(\mathbf{W}^{[2]})^T+\mathbf{B}^{[2]},\\
\hat{\mathbf{p}} &= \sigma(\mathbf{Z}^{[2]}).
\end{aligned}
$$

Here, each row of $\mathbf{W}^{[1]}$ holds the nine weights for one hidden
unit, so $\mathbf{W}^{[1]}\in\mathbb{R}^{3\times9}$. Similarly,
$\mathbf{W}^{[2]}\in\mathbb{R}^{1\times3}$ maps the three hidden
activations to one output score. For this batch, $\mathbf{B}^{[1]}\in
\mathbb{R}^{4\times3}$ and $\mathbf{B}^{[2]}\in\mathbb{R}^{4\times1}$
repeat the learned biases for each observation. The sigmoid converts each
final score to a number between 0 and 1.


In [3]:
def sigmoid(values: np.ndarray) -> np.ndarray:
    """Map output scores to values between zero and one.

    Args:
        values: Array of real-valued output scores.

    Returns:
        Array with the same shape as `values`, with entries in [0, 1].
    """
    return 1 / (1 + np.exp(-values))


def relu(values: np.ndarray) -> np.ndarray:
    """Apply the rectified linear-unit activation elementwise.

    Args:
        values: Array of hidden-layer scores.

    Returns:
        Array with negative entries replaced by zero.
    """
    return np.maximum(values, 0)


def identity(values: np.ndarray) -> np.ndarray:
    """Return hidden-layer scores unchanged.

    Args:
        values: Array of hidden-layer scores.

    Returns:
        The input array, unchanged.
    """
    return values


In [4]:
# A fixed seed makes this random example reproducible. These parameters have
# not been fitted to the gas-turbine labels.
N_HIDDEN_UNITS = 3
rng = np.random.default_rng(4520)

# This follows the slide convention: each row gives one unit's weights.
weights_hidden = rng.normal(
    loc=0.0, scale=0.6, size=(N_HIDDEN_UNITS, len(FEATURES))
)
hidden_bias_per_unit = rng.normal(loc=0.0, scale=0.15, size=N_HIDDEN_UNITS)
weights_output = rng.normal(loc=0.0, scale=0.6, size=(1, N_HIDDEN_UNITS))
output_bias = rng.normal(loc=0.0, scale=0.15, size=1)

# The slide writes B[1] and B[2] with one row for every observation.
bias_hidden = np.broadcast_to(
    hidden_bias_per_unit, (X_batch.shape[0], N_HIDDEN_UNITS)
)
bias_output = np.broadcast_to(output_bias, (X_batch.shape[0], 1))

hidden_weight_table = pd.DataFrame(
    weights_hidden,
    index=["hidden unit 1", "hidden unit 2", "hidden unit 3"],
    columns=FEATURES,
)
hidden_bias_table = pd.DataFrame(
    [hidden_bias_per_unit],
    index=["b[1]"],
    columns=["hidden unit 1", "hidden unit 2", "hidden unit 3"],
)
output_weight_table = pd.DataFrame(
    weights_output,
    index=["output unit"],
    columns=["hidden unit 1", "hidden unit 2", "hidden unit 3"],
)

print("Random W[1] (one row per hidden unit):")
display(hidden_weight_table.round(3))
print("Random b[1] (one bias per hidden unit):")
display(hidden_bias_table.round(3))
print("Random W[2] (one row for the output unit):")
display(output_weight_table.round(3))
print(f"Random b[2]: {output_bias[0]:.3f}")


Random W[1] (one row per hidden unit):


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP
hidden unit 1,-0.140,1.143,0.340,-0.511,0.280,0.809,-0.281,-0.683,0.003
hidden unit 2,0.449,-0.359,-0.493,0.676,-0.351,-0.271,-0.336,-0.931,-0.123
hidden unit 3,-0.263,-1.248,-0.154,0.088,0.046,-0.890,0.571,1.354,-0.033


Random b[1] (one bias per hidden unit):


,hidden unit 1,hidden unit 2,hidden unit 3
b[1],0.154,0.006,0.025


Random W[2] (one row for the output unit):


,hidden unit 1,hidden unit 2,hidden unit 3
output unit,-0.697,-1.349,0.052


Random b[2]: 0.237


### Compute the hidden layer explicitly

The `@` operator performs matrix multiplication. Each row of
$\mathbf{W}^{[1]}$ describes one hidden unit, so we transpose it when
multiplying the input batch. We then add $\mathbf{B}^{[1]}$, which repeats
the three learned hidden biases across the four observations, and apply ReLU
elementwise.


In [5]:
hidden_weighted_sums = X_batch @ weights_hidden.T
hidden_scores_explicit = hidden_weighted_sums + bias_hidden
hidden_activations_explicit = relu(hidden_scores_explicit)

hidden_trace = pd.DataFrame(
    np.hstack(
        [
            hidden_weighted_sums,
            hidden_scores_explicit,
            hidden_activations_explicit,
        ]
    ),
    index=pd.Index(BATCH_ROWS, name="source row"),
    columns=[
        "X(W[1])^T, unit 1",
        "X(W[1])^T, unit 2",
        "X(W[1])^T, unit 3",
        "Z[1], unit 1",
        "Z[1], unit 2",
        "Z[1], unit 3",
        "A[1], unit 1",
        "A[1], unit 2",
        "A[1], unit 3",
    ],
)

print(
    f"X: {X_batch.shape} @ (W[1].T): {weights_hidden.T.shape} "
    f"-> {hidden_weighted_sums.shape}"
)
print(f"B[1]: {bias_hidden.shape} repeats b[1] across the batch.")
hidden_trace.round(3)


X: (4, 9) @ (W[1].T): (9, 3) -> (4, 3)
B[1]: (4, 3) repeats b[1] across the batch.


,"X(W[1])^T, unit 1","X(W[1])^T, unit 2","X(W[1])^T, unit 3","Z[1], unit 1","Z[1], unit 2","Z[1], unit 3","A[1], unit 1","A[1], unit 2","A[1], unit 3"
source row,,,,,,,,,
13311,-0.134,0.217,-0.133,0.020,0.224,-0.108,0.020,0.224,0.0
4066,0.284,0.849,-1.030,0.438,0.855,-1.005,0.438,0.855,0.0
8682,1.604,1.873,-2.973,1.758,1.880,-2.948,1.758,1.880,0.0
5064,0.589,1.086,-1.318,0.743,1.092,-1.293,0.743,1.092,0.0


### Compute the output layer explicitly

The hidden activations now have three columns, one for each hidden unit.
Multiply them by the transposed output weights, add $\mathbf{B}^{[2]}$, and apply
the sigmoid to obtain a probability for each observation.


In [6]:
output_weighted_sums = hidden_activations_explicit @ weights_output.T
output_scores_explicit = output_weighted_sums + bias_output
probabilities_explicit = sigmoid(output_scores_explicit)

output_trace = pd.DataFrame(
    {
        "A[1](W[2])^T": output_weighted_sums.ravel(),
        "Z[2]": output_scores_explicit.ravel(),
        "p-hat": probabilities_explicit.ravel(),
    },
    index=pd.Index(BATCH_ROWS, name="source row"),
)

print(
    f"A[1]: {hidden_activations_explicit.shape} @ (W[2].T): "
    f"{weights_output.T.shape} -> {output_weighted_sums.shape}"
)
print(f"B[2]: {bias_output.shape} repeats b[2] across the batch.")
output_trace.round(3)


A[1]: (4, 3) @ (W[2].T): (3, 1) -> (4, 1)
B[2]: (4, 1) repeats b[2] across the batch.


,A[1](W[2])^T,Z[2],p-hat
source row,,,
13311,-0.316,-0.078,0.480
4066,-1.459,-1.221,0.228
8682,-3.761,-3.524,0.029
5064,-1.991,-1.754,0.148


### Package the calculation in a function

The two cells above are the complete forward pass. Now that each operation
is visible, we can wrap the same sequence in a reusable function. The
function should give exactly the same arrays as the explicit calculation.


In [7]:
def forward_pass(
    *,
    X: np.ndarray,
    weights_hidden: np.ndarray,
    bias_hidden: np.ndarray,
    weights_output: np.ndarray,
    bias_output: np.ndarray,
    activation: Callable[[np.ndarray], np.ndarray],
) -> dict[str, np.ndarray]:
    """Compute one forward pass through a one-hidden-layer network.

    Args:
        X: Input feature matrix with shape (n_observations, n_features).
        weights_hidden: Hidden-layer weights with shape
            (n_hidden_units, n_features).
        bias_hidden: Broadcast hidden biases, with shape
            (n_observations, n_hidden_units).
        weights_output: Output-layer weights with shape
            (1, n_hidden_units).
        bias_output: Broadcast output bias with shape
            (n_observations, 1).
        activation: Elementwise hidden-layer activation function.

    Returns:
        Dictionary containing the hidden scores, hidden activations, output
        scores, and positive-label probabilities.
    """
    hidden_scores = X @ weights_hidden.T + bias_hidden
    hidden_activations = activation(hidden_scores)
    output_scores = hidden_activations @ weights_output.T + bias_output
    probabilities = sigmoid(output_scores)

    return {
        "hidden_scores": hidden_scores,
        "hidden_activations": hidden_activations,
        "output_scores": output_scores,
        "probabilities": probabilities,
    }


relu_outputs = forward_pass(
    X=X_batch,
    weights_hidden=weights_hidden,
    bias_hidden=bias_hidden,
    weights_output=weights_output,
    bias_output=bias_output,
    activation=relu,
)

np.testing.assert_allclose(
    actual=relu_outputs["hidden_scores"], desired=hidden_scores_explicit
)
np.testing.assert_allclose(
    actual=relu_outputs["hidden_activations"],
    desired=hidden_activations_explicit,
)
np.testing.assert_allclose(
    actual=relu_outputs["output_scores"], desired=output_scores_explicit
)
np.testing.assert_allclose(
    actual=relu_outputs["probabilities"], desired=probabilities_explicit
)

shape_table = pd.DataFrame(
    {
        "quantity": [
            "X: input features",
            "W[1]: hidden weights",
            "B[1]: broadcast hidden biases",
            "Z[1]: hidden scores",
            "A[1]: hidden activations",
            "W[2]: output weights",
            "B[2]: broadcast output bias",
            "Z[2]: output scores",
            "p-hat: probabilities",
        ],
        "shape": [
            X_batch.shape,
            weights_hidden.shape,
            bias_hidden.shape,
            relu_outputs["hidden_scores"].shape,
            relu_outputs["hidden_activations"].shape,
            weights_output.shape,
            bias_output.shape,
            relu_outputs["output_scores"].shape,
            relu_outputs["probabilities"].shape,
        ],
    }
)

print("The reusable function matches the explicit calculation.")
shape_table


The reusable function matches the explicit calculation.


,quantity,shape
0,X: input features,"(4, 9)"
1,W[1]: hidden weights,"(3, 9)"
2,B[1]: broadcast hidden biases,"(4, 3)"
3,Z[1]: hidden scores,"(4, 3)"
4,A[1]: hidden activations,"(4, 3)"
5,W[2]: output weights,"(1, 3)"
6,B[2]: broadcast output bias,"(4, 1)"
7,Z[2]: output scores,"(4, 1)"
8,p-hat: probabilities,"(4, 1)"


## 3. Trace one observation through the network

For one observation, each hidden unit first forms a weighted sum of the
nine standardized inputs. ReLU keeps a positive score and replaces a
negative score with zero. The output unit then combines the three hidden
activations into one score and one probability.


In [8]:
OBSERVATION_POSITION = 2
source_row = BATCH_ROWS[OBSERVATION_POSITION]

observation_inputs = pd.DataFrame(
    {
        "raw value": model_data.loc[source_row, FEATURES],
        "standardized value": X_batch[OBSERVATION_POSITION],
    }
)
observation_inputs.index.name = "feature"

trace = pd.DataFrame(
    {
        "hidden unit 1": [
            relu_outputs["hidden_scores"][OBSERVATION_POSITION, 0],
            relu_outputs["hidden_activations"][OBSERVATION_POSITION, 0],
        ],
        "hidden unit 2": [
            relu_outputs["hidden_scores"][OBSERVATION_POSITION, 1],
            relu_outputs["hidden_activations"][OBSERVATION_POSITION, 1],
        ],
        "hidden unit 3": [
            relu_outputs["hidden_scores"][OBSERVATION_POSITION, 2],
            relu_outputs["hidden_activations"][OBSERVATION_POSITION, 2],
        ],
    },
    index=["hidden score, Z[1]", "hidden activation, A[1]"],
)

print(f"Tracing source row {source_row}, whose observed label is {y_batch[OBSERVATION_POSITION]}.")
print("\nInputs used by this observation:")
display(observation_inputs.round(3))
print(
    f"Output score: {relu_outputs['output_scores'][OBSERVATION_POSITION, 0]:.3f}\n"
    f"Predicted positive-label probability: "
    f"{relu_outputs['probabilities'][OBSERVATION_POSITION, 0]:.3f}"
)
trace.round(3)


Tracing source row 8682, whose observed label is 1.

Inputs used by this observation:


,raw value,standardized value
feature,,
AT,7.909,-1.318
AP,1025.900,1.981
AH,88.785,0.754
AFDP,4.088,0.212
GTEP,19.981,-1.330
TIT,1033.900,-2.705
TAT,532.250,-2.042
TEY,110.720,-1.459
CDP,10.633,-1.310


Output score: -3.524
Predicted positive-label probability: 0.029


,hidden unit 1,hidden unit 2,hidden unit 3
"hidden score, Z[1]",1.758,1.88,-2.948
"hidden activation, A[1]",1.758,1.88,0.000


The observed label appears here only to give the input a real context. These
random weights were not trained to match the labels, so this is not a model
evaluation.


## 4. Make predictions for the whole batch

The same matrix operations work for one observation or many. The forward
pass returns one probability for every row of $\mathbf{X}$. The table below
keeps the output trace compact; the preceding section shows all nine inputs
for one of these observations.


In [9]:
batch_predictions = pd.DataFrame(
    {
        "observed label": y_batch,
        "output score": relu_outputs["output_scores"].ravel(),
        "positive-label probability": relu_outputs["probabilities"].ravel(),
    },
    index=pd.Index(BATCH_ROWS, name="source row"),
)
batch_predictions.round(3)


,observed label,output score,positive-label probability
source row,,,
13311,0,-0.078,0.480
4066,0,-1.221,0.228
8682,1,-3.524,0.029
5064,1,-1.754,0.148


## 5. Why the activation function matters

A hidden layer without a nonlinear activation cannot create a more flexible
function. If we replace ReLU with the identity function, the two linear
layers collapse into one effective linear score before the final sigmoid.

We can verify that numerically.


In [10]:
identity_outputs = forward_pass(
    X=X_batch,
    weights_hidden=weights_hidden,
    bias_hidden=bias_hidden,
    weights_output=weights_output,
    bias_output=bias_output,
    activation=identity,
)

effective_weights = weights_output @ weights_hidden
effective_bias = bias_hidden @ weights_output.T + bias_output
collapsed_probabilities = sigmoid(
    X_batch @ effective_weights.T + effective_bias
)

np.testing.assert_allclose(
    actual=identity_outputs["probabilities"],
    desired=collapsed_probabilities,
)

activation_comparison = pd.DataFrame(
    {
        "ReLU hidden layer": relu_outputs["probabilities"].ravel(),
        "identity hidden layer": identity_outputs["probabilities"].ravel(),
        "collapsed linear network": collapsed_probabilities.ravel(),
    },
    index=[f"source row {row}" for row in BATCH_ROWS],
)

print("The assertion passed: the identity network matches one linear score plus sigmoid.")
activation_comparison.round(3)


The assertion passed: the identity network matches one linear score plus sigmoid.


,ReLU hidden layer,identity hidden layer,collapsed linear network
source row 13311,0.480,0.479,0.479
source row 4066,0.228,0.219,0.219
source row 8682,0.029,0.025,0.025
source row 5064,0.148,0.139,0.139


The ReLU network does not collapse in this way because it changes values
depending on whether each hidden score falls above or below zero. That
nonlinearity lets neural networks represent nonlinear relationships.


## 6. Count the parameters

For $p$ input features, $h$ hidden units, and one output unit, this network
has $ph+h+h+1$ parameters: hidden weights and biases, then output weights
and bias. Increasing hidden-layer width increases the model's capacity and
also increases the number of quantities that later need to be learned.


In [11]:
def count_parameters(*, n_features: int, n_hidden_units: int) -> int:
    """Count parameters in a one-hidden-layer binary classifier.

    Args:
        n_features: Number of input features.
        n_hidden_units: Number of hidden units.

    Returns:
        Total number of weights and biases in the network.
    """
    hidden_layer = n_features * n_hidden_units + n_hidden_units
    output_layer = n_hidden_units + 1
    return hidden_layer + output_layer


parameter_counts = pd.DataFrame(
    {
        "input features": [9, 9, 9],
        "hidden units": [3, 4, 8],
        "parameters": [
            count_parameters(n_features=9, n_hidden_units=3),
            count_parameters(n_features=9, n_hidden_units=4),
            count_parameters(n_features=9, n_hidden_units=8),
        ],
    }
)
parameter_counts


,input features,hidden units,parameters
0,9,3,34
1,9,4,45
2,9,8,89


## Takeaways

- Forward propagation carries each observation through weighted sums and
  activation functions to produce a prediction.
- With one row per observation, matrix shapes reveal how an entire batch can
  move through the network at once.
- ReLU allows stacked hidden layers to avoid collapsing into one linear model.
- The next question is how to choose the weights and biases from data. That
  requires a loss function and backpropagation.

[← Return to Neural networks](index.qmd)
